# MMCT Ingestion Pipeline: Usage Guide

This notebook demonstrates how to use the refactored **MMCT Ingestion Pipeline** to process video files. The new architecture is built on a modular, step-based framework that allows for transparent execution, robust data sharing between steps, and hierarchical graph construction.

### 🚀 Key Features
- **Step-Based Execution**: Individual components (Transcription, Keyframes, Temporal Graph) are isolated and reusable.
- **Hierarchical Knowledge Graph**: Automatically builds nodes for ChapterGroups, Chapters, Events, and Objects.
- **Multimodal Extraction**: Combines transcript data with visual frame analysis for deeper context.
- **Provider Agnostic**: Easily switch between Azure OpenAI, local embeddings, and different storage backends.

---

### 🛠️ Prerequisites
Ensure you have the required environment variables set in your `.env` file:
- `LLM_ENDPOINT`, `LLM_API_KEY`
- `EMBEDDING_SERVICE_ENDPOINT`, `EMBEDDING_SERVICE_API_KEY`
- `STORAGE_ACCOUNT_NAME`, `STORAGE_ACCESS_KEY`
- `NEO4J_URI`, `NEO4J_PASSWORD` (optional for local testing)

## 1. Setup and Imports

In [ ]:
import os
import asyncio
import nest_asyncio
from loguru import logger

nest_asyncio.apply()

from mmct.video_pipeline import IngestionPipeline, Languages
from mmct.video_pipeline.utils.helper import get_file_hash
from config.provider_config import get_ingestion_providers
from mmct.providers.azure_providers import (
    AzureLLMProvider,
    AzureStorageProvider,
    WhisperTranscriptionProvider,
)

## 2. Video and Metadata Input

Provide the details for the video you wish to process. If you already have a transcript, the pipeline will skip the transcription step.

In [ ]:
# Input Video Details
video_path = "path/to/your/video.mp4"
video_id = None # Leave None to auto-generate from file hash
url = "https://example.com/video" # Optional metadata

# Language Setting (Required if no transcript is provided)
source_language = Languages.ENGLISH_UNITED_STATES

# Optional: Path to an existing .srt file
transcript_path = None 

if not video_id:
    video_id = await get_file_hash(video_path)
    
print(f"Processing Video ID: {video_id}")

## 3. Provider Configuration

The pipeline requires a bundle of providers for LLM, Embeddings, Storage, and Transcription. You can load these automatically from your environment variables.

In [ ]:
# Option A: Simple setup from environment variables (.env)
providers = get_ingestion_providers()

"""
# Option B: Manual setup for custom configurations
providers = IngestionProviders(
    llm_provider=AzureLLMProvider(
        endpoint=os.getenv("LLM_ENDPOINT"),
        deployment_name="gpt-4o",
        api_key=os.getenv("LLM_API_KEY")
    ),
    transcription_provider=WhisperTranscriptionProvider(...),
    storage_provider=AzureStorageProvider(...)
)
"""
print("Providers initialized successfully.")

## 4. Run the Ingestion Pipeline

The `IngestionPipeline` will orchestrate the execution across all steps. By default, it follows the `temporal_graph_ingestion` workflow.

In [ ]:
ingestion = IngestionPipeline(
    video_path=video_path,
    video_id=video_id,
    provider=providers,
    language=source_language,
    transcript_path=transcript_path,
    url=url,
    verbosity=1, # 0: Progress Bar, 1: Info Logs, 2: Debug Logs
    save_local_report=True
)

print("Starting Ingestion Lifecycle...")
report = await ingestion.run()

if report.status == "completed":
    print(f"🎉 Pipeline finished successfully in {report.total_duration_seconds:.2f}s")
else:
    print("❌ Pipeline encountered errors during execution.")

## 5. Inspect Results

You can inspect the generated data directly from memory or check the `media/` folder for exported artifacts.

In [ ]:
# Example: Check extracted chapters
if report.status == "completed":
    # Note: data can be retrieved from the report if implemented, 
    # or check the local media folder for the JSON exports.
    export_path = "./media/export/"
    print(f"Results exported to {os.path.abspath(export_path)}")